<a href="https://colab.research.google.com/github/DalpesCS50/trab_estagio4_health-insurance-claims-prediction/blob/main/notebooks/Metodos_de_Reamostragem_health_insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Métodos de Reamostragem — HEALTH INSURANCE

Adaptação do roteiro do professor à base HEALTH INSURANCE para o problema de regressão com resposta claim.
- Divisão treino/teste (70%/30%)
- K-fold com 5 grupos
- K-fold repetido (seção denominada Monte Carlo no roteiro)
- Bootstrap com validação fora da amostra (out-of-bag)

Esta etapa demonstra a formação das amostras; não treina modelos nem calcula métricas. Os dados são carregados em sua forma bruta, sem reutilizar o PCA ajustado na base completa. Na futura modelagem, imputação, codificação, escalonamento e eventual PCA deverão ser ajustados somente no treino de cada divisão. O teste de 30% ficará reservado para a avaliação final.

Documentação: https://scikit-learn.org/stable/modules/cross_validation.html

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, RepeatedKFold
from sklearn.utils import resample

## Carregar a base HEALTH INSURANCE

Usamos o mesmo arquivo de dados do projeto de pré-processamento. Mantemos todas as 12 preditoras, incluindo job_title, e separamos claim como resposta em um DataFrame de uma coluna, seguindo a estrutura do professor.

## Origem dos dados

O arquivo é lido diretamente do repositório do projeto; não é necessário montar o Google Drive.

## Função de carregamento

In [2]:
def carrega_dados():
    """Retorna as preditoras brutas X e a resposta y da HEALTH INSURANCE."""
    url = "https://raw.githubusercontent.com/DalpesCS50/trab_estagio4_health-insurance-claims-prediction/refs/heads/main/data/healthinsurance.csv"
    df = pd.read_csv(url)
    assert 'claim' in df.columns
    assert df['claim'].notna().all(), "A resposta contém valores ausentes."
    X = df.drop(columns='claim').copy()
    y = df[['claim']].copy()
    return X, y

In [3]:
X, y = carrega_dados()
print('Dimensão de X:', X.shape)
print('Dimensão de y:', y.shape)
print('Ausências nas preditoras:')
display(X.isna().sum())

Dimensão de X: (15000, 12)
Dimensão de y: (15000, 1)
Ausências nas preditoras:


,0
age,396
sex,0
weight,0
bmi,956
hereditary_diseases,0
no_of_dependents,0
smoker,0
city,0
bloodpressure,0
diabetes,0


### Carregamento dos dados para reamostragem

A base HEALTH INSURANCE foi separada em `X`, com 15.000 observações e 12 variáveis preditoras, e `y`, com 15.000 observações e uma coluna correspondente à resposta `claim`.

As preditoras permanecem em sua forma original, incluindo `job_title` e os valores ausentes: 396 em `age` e 956 em `bmi`. Na futura modelagem, o tratamento dessas ausências será ajustado somente nos dados de treino de cada divisão e aplicado à validação ou ao teste correspondente.

In [4]:
X

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title
0,60.0,male,64,24.3,NoDisease,1,0,NewYork,72,0,0,Actor
1,49.0,female,75,22.6,NoDisease,1,0,Boston,78,1,1,Engineer
2,32.0,female,64,17.8,Epilepsy,2,1,Phildelphia,88,1,1,Academician
3,61.0,female,53,36.4,NoDisease,1,1,Pittsburg,72,1,0,Chef
4,19.0,female,50,20.6,NoDisease,0,0,Buffalo,82,1,0,HomeMakers
...,...,...,...,...,...,...,...,...,...,...,...,...
14995,39.0,male,49,28.3,NoDisease,1,1,Florence,54,1,0,FilmMaker
14996,39.0,male,74,29.6,NoDisease,4,0,Miami,64,1,0,Student
14997,20.0,male,62,33.3,NoDisease,0,0,Tampa,52,1,0,FashionDesigner
14998,52.0,male,88,36.7,NoDisease,0,0,PanamaCity,70,1,0,Farmer


In [5]:
y

,claim
0,13112.6
1,9567.0
2,32734.2
3,48517.6
4,1731.7
...,...
14995,21082.2
14996,7512.3
14997,1391.5
14998,9144.6


## Divisão treino/teste

Mantemos a escolha do professor: 70% para treino e 30% para teste, com random_state=1. Para 15.000 observações, esperamos 10.500 no treino e 4.500 no teste. Esta divisão pertence à etapa de reamostragem; não reutiliza a antiga divisão 80/20.

As ausências e as categorias ainda não foram tratadas. Na modelagem, as transformações serão aprendidas no treino e aplicadas ao teste.

In [6]:
X.columns

Index(['age', 'sex', 'weight', 'bmi', 'hereditary_diseases',
       'no_of_dependents', 'smoker', 'city', 'bloodpressure', 'diabetes',
       'regular_ex', 'job_title'],
      dtype='object')

In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  14604 non-null  float64
 1   sex                  15000 non-null  object 
 2   weight               15000 non-null  int64  
 3   bmi                  14044 non-null  float64
 4   hereditary_diseases  15000 non-null  object 
 5   no_of_dependents     15000 non-null  int64  
 6   smoker               15000 non-null  int64  
 7   city                 15000 non-null  object 
 8   bloodpressure        15000 non-null  int64  
 9   diabetes             15000 non-null  int64  
 10  regular_ex           15000 non-null  int64  
 11  job_title            15000 non-null  object 
dtypes: float64(2), int64(6), object(4)
memory usage: 1.4+ MB


In [8]:
X.shape[0] * 0.7

10500.0

In [9]:
# definir a `%` de treinamento
perc_train = 0.7
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=perc_train, random_state=1)

In [10]:
X_train

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title
4834,49.0,female,87,31.9,NoDisease,5,0,Charlotte,72,1,0,Clerks
12171,22.0,female,60,23.2,NoDisease,0,0,FallsCity,94,1,1,Student
3502,61.0,male,66,32.3,NoDisease,2,0,Cincinnati,65,1,0,FilmMaker
11546,51.0,female,51,34.2,NoDisease,1,0,SantaFe,0,1,1,Singer
6950,19.0,male,47,27.3,NoDisease,2,0,WashingtonDC,62,1,0,Student
...,...,...,...,...,...,...,...,...,...,...,...,...
905,37.0,female,76,38.4,HeartDisease,0,1,Lincoln,86,1,0,ITProfessional
5192,25.0,female,67,42.1,NoDisease,1,0,SanDeigo,70,0,0,Analyst
12172,25.0,male,47,26.7,NoDisease,4,0,GrandForks,74,1,1,Farmer
235,61.0,male,69,23.7,NoDisease,0,0,SantaFe,72,1,1,Singer


In [11]:
X.shape[0] * 0.7

10500.0

In [12]:
X.shape[0] * 0.3

4500.0

In [13]:
X_train.shape[0]

10500

In [14]:
X_test.shape[0]

4500

In [15]:
# mostrar os índices selecionados
print('TREINAMENTO')
print(X_train.index)
print(y_train.index)
print('\n------------------------------------------------------------------------\n')
print('TESTE')
print(X_test.index)
print(y_test.index)

TREINAMENTO
Index([ 4834, 12171,  3502, 11546,  6950,  8808,  3259,  1125,  6672,  2617,
       ...
         144,  5056,  2895,  7813, 10955,   905,  5192, 12172,   235, 13349],
      dtype='int64', length=10500)
Index([ 4834, 12171,  3502, 11546,  6950,  8808,  3259,  1125,  6672,  2617,
       ...
         144,  5056,  2895,  7813, 10955,   905,  5192, 12172,   235, 13349],
      dtype='int64', length=10500)

------------------------------------------------------------------------

TESTE
Index([ 7576, 10509,  4253,  5150,   506,  1622, 13684,  2145,  9296,  6900,
       ...
       12303, 10153, 11244, 13559, 12081,  5682, 11109,  7443, 10077, 13653],
      dtype='int64', length=4500)
Index([ 7576, 10509,  4253,  5150,   506,  1622, 13684,  2145,  9296,  6900,
       ...
       12303, 10153, 11244, 13559, 12081,  5682, 11109,  7443, 10077, 13653],
      dtype='int64', length=4500)


In [16]:
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(X)
print('Índices de X e y alinhados e conjuntos separados.')
print('Quantidade de observações para treinamento:', X_train.shape)
print('Quantidade de observações para teste:', X_test.shape)

Índices de X e y alinhados e conjuntos separados.
Quantidade de observações para treinamento: (10500, 12)
Quantidade de observações para teste: (4500, 12)


### Resultado da divisão treino/teste

A base HEALTH INSURANCE foi dividida em 10.500 observações para treinamento (70%) e 4.500 para teste (30%), mantendo as 12 variáveis preditoras.

As verificações confirmaram que os índices das preditoras e da resposta `claim` estão alinhados em ambos os conjuntos, sem sobreposição entre treino e teste e sem perda de observações.

A divisão utilizou `random_state=1` para permitir sua reprodução. O conjunto de teste ficará reservado para a avaliação final; os métodos de reamostragem serão aplicados ao conjunto de treinamento.

In [17]:
X_train.head()

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title
4834,49.0,female,87,31.9,NoDisease,5,0,Charlotte,72,1,0,Clerks
12171,22.0,female,60,23.2,NoDisease,0,0,FallsCity,94,1,1,Student
3502,61.0,male,66,32.3,NoDisease,2,0,Cincinnati,65,1,0,FilmMaker
11546,51.0,female,51,34.2,NoDisease,1,0,SantaFe,0,1,1,Singer
6950,19.0,male,47,27.3,NoDisease,2,0,WashingtonDC,62,1,0,Student


In [18]:
X_test.head()

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title
7576,57.0,male,72,34.0,NoDisease,0,0,GrandForks,48,1,0,Politician
10509,63.0,female,69,25.1,NoDisease,0,0,Kingsport,0,1,1,Manager
4253,45.0,female,80,36.3,NoDisease,2,0,SilverCity,54,1,0,Clerks
5150,37.0,male,67,46.5,NoDisease,3,0,Carlsbad,84,1,0,Manager
506,19.0,female,54,NaN,Obesity,0,1,Baltimore,94,0,0,Student


## K-fold

Primeiro, demonstramos 5 folds em 20 posições: 16 para treino e 4 para validação em cada divisão. Depois, aplicamos o método somente às 10.500 observações de treino da HEALTH INSURANCE: 8.400 para treino interno e 2.100 para validação por fold.

Preservamos KFold(n_splits=5) sem embaralhamento, como no roteiro. Os índices retornados são posições e devem ser usados com .iloc. O teste externo não participa dos folds.

In [19]:
lista = np.arange(0, 20)
lista

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19])

In [20]:
# vamos testar em uma lista com 20 observações primeiro
lista = np.arange(0,20)
print(lista)
print('')


# selecionar 5 grupos
kfold_cv = KFold(n_splits=5)
print('quantidade de k:',kfold_cv.get_n_splits())
print('')

# realizar a iteração para mostrar os grupos gerados
for train_idx, val_idx in kfold_cv.split(lista):
    print("TREINAMENTO:", train_idx, "| VALIDAÇÃO:", val_idx)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]

quantidade de k: 5

TREINAMENTO: [ 4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19] | VALIDAÇÃO: [0 1 2 3]
TREINAMENTO: [ 0  1  2  3  8  9 10 11 12 13 14 15 16 17 18 19] | VALIDAÇÃO: [4 5 6 7]
TREINAMENTO: [ 0  1  2  3  4  5  6  7 12 13 14 15 16 17 18 19] | VALIDAÇÃO: [ 8  9 10 11]
TREINAMENTO: [ 0  1  2  3  4  5  6  7  8  9 10 11 16 17 18 19] | VALIDAÇÃO: [12 13 14 15]
TREINAMENTO: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] | VALIDAÇÃO: [16 17 18 19]


### Demonstração do K-fold

Na lista de 20 posições, o K-fold formou cinco divisões, cada uma com 16 posições para treinamento e quatro para validação.

Cada posição foi utilizada uma vez na validação e quatro vezes no treinamento. Como não houve embaralhamento, os grupos de validação foram formados por posições consecutivas.

A seguir, o mesmo procedimento será aplicado ao conjunto de treinamento da base HEALTH INSURANCE, mantendo o teste externo reservado.

In [21]:
X.iloc[[0, 1, 2]]

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title
0,60.0,male,64,24.3,NoDisease,1,0,NewYork,72,0,0,Actor
1,49.0,female,75,22.6,NoDisease,1,0,Boston,78,1,1,Engineer
2,32.0,female,64,17.8,Epilepsy,2,1,Phildelphia,88,1,1,Academician


In [22]:
X_train.iloc[[4, 5, 6, 7, 8]]

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title
6950,19.0,male,47,27.3,NoDisease,2,0,WashingtonDC,62,1,0,Student
8808,49.0,male,61,37.5,NoDisease,2,0,Indianapolis,88,1,0,Manager
3259,28.0,female,59,29.3,Cancer,2,1,LosAngeles,74,1,1,Engineer
1125,39.0,male,74,29.6,NoDisease,4,0,Chicago,64,1,0,Student
6672,32.0,male,95,27.8,NoDisease,1,0,SanJose,50,1,0,Engineer


In [23]:
# Agora vamos realizar esse k-fold no nosso conjunto de dados
for train_idx, val_idx in kfold_cv.split(X_train):
    print("TREINAMENTO:", train_idx, "| VALIDAÇÃO:", val_idx)

TREINAMENTO: [ 2100  2101  2102 ... 10497 10498 10499] | VALIDAÇÃO: [   0    1    2 ... 2097 2098 2099]
TREINAMENTO: [    0     1     2 ... 10497 10498 10499] | VALIDAÇÃO: [2100 2101 2102 ... 4197 4198 4199]
TREINAMENTO: [    0     1     2 ... 10497 10498 10499] | VALIDAÇÃO: [4200 4201 4202 ... 6297 6298 6299]
TREINAMENTO: [    0     1     2 ... 10497 10498 10499] | VALIDAÇÃO: [6300 6301 6302 ... 8397 8398 8399]
TREINAMENTO: [   0    1    2 ... 8397 8398 8399] | VALIDAÇÃO: [ 8400  8401  8402 ... 10497 10498 10499]


### K-fold aplicado à base HEALTH INSURANCE

O K-fold foi aplicado às 10.500 observações do conjunto de treinamento, formando cinco divisões. Em cada divisão, 8.400 observações são destinadas ao treinamento interno e 2.100 à validação.

Cada observação participa da validação exatamente uma vez. Os grupos são definidos por posições consecutivas dentro de `X_train`, pois o K-fold não utiliza embaralhamento nesta configuração.

As 4.500 observações do teste externo não participam dessas divisões.

In [24]:
X_train_lista, X_val_lista = [], []
y_train_lista, y_val_lista = [], []

for train_idx, val_idx in kfold_cv.split(X_train):
    X_train_lista.append(X_train.iloc[train_idx].copy())
    X_val_lista.append(X_train.iloc[val_idx].copy())
    y_train_lista.append(y_train.iloc[train_idx].copy())
    y_val_lista.append(y_train.iloc[val_idx].copy())
    assert X_train_lista[-1].index.equals(y_train_lista[-1].index)
    assert X_val_lista[-1].index.equals(y_val_lista[-1].index)

for fold, (treino, validacao) in enumerate(zip(X_train_lista, X_val_lista), 1):
    print(f'Fold {fold}: treino {treino.shape}, validação {validacao.shape}')

Fold 1: treino (8400, 12), validação (2100, 12)
Fold 2: treino (8400, 12), validação (2100, 12)
Fold 3: treino (8400, 12), validação (2100, 12)
Fold 4: treino (8400, 12), validação (2100, 12)
Fold 5: treino (8400, 12), validação (2100, 12)


In [25]:
len(X_train_lista)

5

In [26]:
X_train_lista[4]

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title
4834,49.0,female,87,31.9,NoDisease,5,0,Charlotte,72,1,0,Clerks
12171,22.0,female,60,23.2,NoDisease,0,0,FallsCity,94,1,1,Student
3502,61.0,male,66,32.3,NoDisease,2,0,Cincinnati,65,1,0,FilmMaker
11546,51.0,female,51,34.2,NoDisease,1,0,SantaFe,0,1,1,Singer
6950,19.0,male,47,27.3,NoDisease,2,0,WashingtonDC,62,1,0,Student
...,...,...,...,...,...,...,...,...,...,...,...,...
2357,44.0,male,54,34.3,NoDisease,1,0,Newport,80,1,0,Student
3395,19.0,female,45,29.8,NoDisease,0,0,Houston,66,0,0,HomeMakers
11040,24.0,male,65,40.2,NoDisease,0,1,SantaRosa,60,1,0,Clerks
7741,36.0,male,94,41.9,NoDisease,3,1,Georgia,68,1,1,Dancer


In [27]:
# agora eu posso extrair os dados (opcional)
X_train_1, X_train_2, X_train_3, X_train_4, X_train_5 = X_train_lista
X_val_1, X_val_2, X_val_3, X_val_4, X_val_5 = X_val_lista

y_train_1, y_train_2, y_train_3, y_train_4, y_train_5 = y_train_lista
y_val_1, y_val_2, y_val_3, y_val_4, y_val_5 = y_val_lista

# mostrar
display(X_train_1.head())
display(y_train_1.head())

,age,sex,weight,bmi,hereditary_diseases,no_of_dependents,smoker,city,bloodpressure,diabetes,regular_ex,job_title
7539,34.0,female,51,33.3,NoDisease,1,0,Worcester,0,0,0,CA
12748,NaN,female,74,21.9,NoDisease,0,1,SantaFe,80,0,0,Beautician
10444,34.0,female,59,33.7,NoDisease,1,0,IowaCity,60,1,1,HouseKeeper
6128,55.0,male,88,32.7,NoDisease,1,0,Oklahoma,0,1,0,CEO
8469,18.0,male,65,34.1,NoDisease,0,0,Raleigh,90,1,0,Student


,claim
7539,5594.9
12748,16115.3
10444,5012.5
6128,10807.5
8469,1137.0


## K-fold repetido — seção Monte Carlo do roteiro

O código do professor usa RepeatedKFold: 5 folds repetidos 10 vezes, totalizando 50 divisões. Trata-se de K-fold repetido, que difere da validação Monte Carlo por divisões aleatórias independentes (ShuffleSplit). Mantemos o método implementado no roteiro e esclarecemos sua denominação.

Exibimos primeiro as divisões na lista de 20 posições e depois os tamanhos das divisões na base de treino HEALTH INSURANCE.

In [28]:
lista = np.arange(20)
mc_cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=1)
print('Quantidade de divisões:', mc_cv.get_n_splits())
for train_idx, val_idx in mc_cv.split(lista):
    print('TREINAMENTO:', train_idx, '| VALIDAÇÃO:', val_idx)

print('\nK-fold repetido no treino HEALTH INSURANCE:')
for divisao, (train_idx, val_idx) in enumerate(mc_cv.split(X_train), 1):
    assert set(train_idx).isdisjoint(val_idx)
    print(f'Divisão {divisao}: treino {len(train_idx)}, validação {len(val_idx)}')

Quantidade de divisões: 50
TREINAMENTO: [ 0  1  2  4  5  7  8  9 11 12 13 14 15 17 18 19] | VALIDAÇÃO: [ 3  6 10 16]
TREINAMENTO: [ 0  1  3  5  6  7  8  9 10 11 12 13 15 16 18 19] | VALIDAÇÃO: [ 2  4 14 17]
TREINAMENTO: [ 2  3  4  5  6  8  9 10 11 12 14 15 16 17 18 19] | VALIDAÇÃO: [ 0  1  7 13]
TREINAMENTO: [ 0  1  2  3  4  5  6  7  8 10 11 12 13 14 16 17] | VALIDAÇÃO: [ 9 15 18 19]
TREINAMENTO: [ 0  1  2  3  4  6  7  9 10 13 14 15 16 17 18 19] | VALIDAÇÃO: [ 5  8 11 12]
TREINAMENTO: [ 0  1  2  4  5  6  7  9 10 11 12 13 14 17 18 19] | VALIDAÇÃO: [ 3  8 15 16]
TREINAMENTO: [ 0  1  2  3  4  7  8  9 10 11 12 13 14 15 16 18] | VALIDAÇÃO: [ 5  6 17 19]
TREINAMENTO: [ 2  3  4  5  6  7  8 10 11 13 14 15 16 17 18 19] | VALIDAÇÃO: [ 0  1  9 12]
TREINAMENTO: [ 0  1  3  5  6  8  9 10 11 12 14 15 16 17 18 19] | VALIDAÇÃO: [ 2  4  7 13]
TREINAMENTO: [ 0  1  2  3  4  5  6  7  8  9 12 13 15 16 17 19] | VALIDAÇÃO: [10 11 14 18]
TREINAMENTO: [ 0  1  3  5  6  7  8  9 10 11 13 14 15 16 17 19] | VALIDAÇÃ

### Resultado do K-fold repetido

O método `RepeatedKFold` foi configurado com cinco folds e dez repetições, utilizando `random_state=1`, totalizando 50 divisões.

Aplicado às 10.500 observações de treinamento da base HEALTH INSURANCE, cada divisão contém 8.400 observações para treino interno e 2.100 para validação. Em cada repetição de cinco folds, cada observação participa da validação uma vez; ao longo das dez repetições, participa dez vezes.

As 4.500 observações do teste externo permanecem reservadas. Nesta etapa, foram demonstradas as divisões, sem treinamento de modelos ou cálculo de desempenho.

## Validação Monte Carlo com ShuffleSplit

Utilizamos `ShuffleSplit` para gerar 50 divisões aleatórias do conjunto de treinamento da HEALTH INSURANCE. Cada divisão utiliza 80% para treino interno e 20% para validação, correspondentes a 8.400 e 2.100 observações.

Dentro de cada divisão, treino e validação não compartilham observações. Entre divisões, uma observação pode participar da validação várias vezes ou nenhuma vez, sem a cobertura uniforme garantida pelo K-fold repetido.

As 4.500 observações do teste externo permanecem reservadas. Esta seção complementa o código do professor e demonstra o particionamento, sem treinar modelos. Na modelagem, as transformações deverão ser ajustadas somente no treino interno de cada divisão.

In [33]:
from sklearn.model_selection import ShuffleSplit

monte_carlo_cv = ShuffleSplit(
    n_splits=50,
    test_size=0.2,
    random_state=1
)

monte_carlo_indices = list(monte_carlo_cv.split(X_train))
contagem_validacao = np.zeros(len(X_train), dtype=int)

print("Quantidade de divisões Monte Carlo:", len(monte_carlo_indices))

for divisao, (idx_treino, idx_val) in enumerate(monte_carlo_indices, 1):
    X_mc_treino = X_train.iloc[idx_treino]
    y_mc_treino = y_train.iloc[idx_treino]
    X_mc_val = X_train.iloc[idx_val]
    y_mc_val = y_train.iloc[idx_val]

    assert set(idx_treino).isdisjoint(idx_val)
    assert len(idx_treino) + len(idx_val) == len(X_train)
    assert X_mc_treino.index.equals(y_mc_treino.index)
    assert X_mc_val.index.equals(y_mc_val.index)

    contagem_validacao[idx_val] += 1

    print(
        f"Divisão {divisao}: treino {len(idx_treino)}, "
        f"validação {len(idx_val)}"
    )

print("Participações na validação por observação:")
print("Mínimo:", contagem_validacao.min())
print("Máximo:", contagem_validacao.max())
print("Média:", contagem_validacao.mean())

Quantidade de divisões Monte Carlo: 50
Divisão 1: treino 8400, validação 2100
Divisão 2: treino 8400, validação 2100
Divisão 3: treino 8400, validação 2100
Divisão 4: treino 8400, validação 2100
Divisão 5: treino 8400, validação 2100
Divisão 6: treino 8400, validação 2100
Divisão 7: treino 8400, validação 2100
Divisão 8: treino 8400, validação 2100
Divisão 9: treino 8400, validação 2100
Divisão 10: treino 8400, validação 2100
Divisão 11: treino 8400, validação 2100
Divisão 12: treino 8400, validação 2100
Divisão 13: treino 8400, validação 2100
Divisão 14: treino 8400, validação 2100
Divisão 15: treino 8400, validação 2100
Divisão 16: treino 8400, validação 2100
Divisão 17: treino 8400, validação 2100
Divisão 18: treino 8400, validação 2100
Divisão 19: treino 8400, validação 2100
Divisão 20: treino 8400, validação 2100
Divisão 21: treino 8400, validação 2100
Divisão 22: treino 8400, validação 2100
Divisão 23: treino 8400, validação 2100
Divisão 24: treino 8400, validação 2100
Divisão 25

### Resultado da validação Monte Carlo

Na HEALTH INSURANCE, foram realizadas 50 divisões aleatórias do conjunto de treinamento, cada uma com 8.400 observações para treino interno e 2.100 para validação.

As verificações confirmaram a separação dos grupos e o alinhamento entre as preditoras e a resposta `claim`.

Nesta execução, cada observação participou da validação entre duas e 23 vezes, com média de dez participações. Essa variação caracteriza a diferença em relação ao K-fold repetido, que garante dez participações por observação na configuração utilizada.

As 4.500 observações do teste externo permaneceram reservadas. Não foram treinados modelos nem calculadas métricas nesta etapa.

## Bootstrap

Em cada uma das 50 repetições, sorteamos com reposição tantas posições quanto existem no conjunto de entrada. Uma observação pode aparecer várias vezes no treino. As posições não sorteadas formam a validação out-of-bag, cujo tamanho varia.

Adaptamos a classe do professor para usar um gerador aleatório local e identificar as posições não sorteadas sem a busca repetida em listas. Na aplicação HEALTH INSURANCE, o método utiliza apenas o treino externo.

In [34]:
class bootstrap:
    def __init__(self, n_splits, random_state):
        self.n_splits = n_splits
        self.random_state = random_state

    def get_n_splits(self):
        return self.n_splits

    def split(self, X):
        rng = np.random.RandomState(self.random_state)
        idx = np.arange(X.shape[0])
        for _ in range(self.n_splits):
            idx_train = resample(idx, replace=True, n_samples=len(idx), random_state=rng)
            idx_val = np.setdiff1d(idx, np.unique(idx_train))
            yield idx_train, idx_val

In [35]:
# vamos usar a lista e geraremos 50 amostras bootstrap
boot = bootstrap(n_splits=50, random_state=1)
print("Quantidade de amostras bootstrap:", boot.get_n_splits())

for id_train, id_test in boot.split(lista):
    print(f"TREINAMENTO: {id_train} | VALIDAÇÃO: {id_test}")

Quantidade de amostras bootstrap: 50
TREINAMENTO: [ 5 11 12  8  9 11  5 15  0 16  1 12  7 13  6 18  5 18 11 10] | VALIDAÇÃO: [ 2  3  4 14 17 19]
TREINAMENTO: [14 18  4  9 17  0 13  9  9  7  1  0 17  8 13 19 15 10  8  7] | VALIDAÇÃO: [ 2  3  5  6 11 12 16]
TREINAMENTO: [ 3  6 17  3  4 17 11 12 16 13 19  9 18 15  0  4 15  2  7  8] | VALIDAÇÃO: [ 1  5 10 14]
TREINAMENTO: [ 9  3  7  4  5 19  6  8  0  2 10 15 15  7 19 10 14  0  1 17] | VALIDAÇÃO: [11 12 13 16 18]
TREINAMENTO: [13  3  0 13  6  6  2 12 11  7 13  8 11 12 11  4  7  7 13  4] | VALIDAÇÃO: [ 1  5  9 10 14 15 16 17 18 19]
TREINAMENTO: [16 18  0 13 10 17  7 10  0 12  1  9 18 19  1  2 12  7  2 10] | VALIDAÇÃO: [ 3  4  5  6  8 11 14 15]
TREINAMENTO: [ 0  9 18  6  2  7  7 16 15  5 17 10 16  5 13 17 18  1 10  0] | VALIDAÇÃO: [ 3  4  8 11 12 14 19]
TREINAMENTO: [ 7  0 19 17 14 13 11  6 13 15  9  2  7  5  4  5  8 13 17 17] | VALIDAÇÃO: [ 1  3 10 12 16 18]
TREINAMENTO: [15 13  8 14 13 16 10 13  3  2 14 14  0 11  3  1  2 19 16  0] | VALIDAÇ

### Demonstração do bootstrap

Foram geradas 50 amostras bootstrap a partir de uma lista de 20 posições. Cada amostra contém 20 sorteios com reposição, permitindo que uma mesma posição apareça várias vezes no treinamento.

As posições não sorteadas compõem a validação out-of-bag (OOB). Por esse motivo, a quantidade de posições únicas no treino e o tamanho da validação variam entre as amostras.

A seguir, o procedimento será aplicado ao conjunto de treinamento da base HEALTH INSURANCE, mantendo o teste externo reservado.

In [36]:
for repeticao, (id_train, id_val) in enumerate(boot.split(X_train), 1):
    X_boot = X_train.iloc[id_train]
    y_boot = y_train.iloc[id_train]
    X_oob = X_train.iloc[id_val]
    y_oob = y_train.iloc[id_val]
    assert X_boot.index.equals(y_boot.index)
    assert X_oob.index.equals(y_oob.index)
    assert set(id_train).isdisjoint(id_val)
    print(f'Amostra {repeticao}: treino {len(id_train)}, '
          f'observações únicas {len(np.unique(id_train))}, validação OOB {len(id_val)}')

Amostra 1: treino 10500, observações únicas 6604, validação OOB 3896
Amostra 2: treino 10500, observações únicas 6615, validação OOB 3885
Amostra 3: treino 10500, observações únicas 6574, validação OOB 3926
Amostra 4: treino 10500, observações únicas 6682, validação OOB 3818
Amostra 5: treino 10500, observações únicas 6646, validação OOB 3854
Amostra 6: treino 10500, observações únicas 6663, validação OOB 3837
Amostra 7: treino 10500, observações únicas 6614, validação OOB 3886
Amostra 8: treino 10500, observações únicas 6631, validação OOB 3869
Amostra 9: treino 10500, observações únicas 6622, validação OOB 3878
Amostra 10: treino 10500, observações únicas 6663, validação OOB 3837
Amostra 11: treino 10500, observações únicas 6700, validação OOB 3800
Amostra 12: treino 10500, observações únicas 6655, validação OOB 3845
Amostra 13: treino 10500, observações únicas 6594, validação OOB 3906
Amostra 14: treino 10500, observações únicas 6583, validação OOB 3917
Amostra 15: treino 10500, obs

### Resultado do bootstrap na HEALTH INSURANCE

Foram geradas 50 amostras bootstrap a partir das 10.500 observações do conjunto de treinamento. Cada amostra contém 10.500 sorteios com reposição, preservando as repetições de registros.

A quantidade de observações únicas no treino variou de 6.574 a 6.711. As observações não sorteadas formaram a validação out-of-bag (OOB), cujo tamanho variou de 3.789 a 3.926.

Em cada repetição, as observações únicas sorteadas e as observações OOB totalizam as 10.500 observações disponíveis, sem sobreposição entre esses grupos. As 4.500 observações do teste externo permaneceram reservadas.

Esta etapa demonstra a formação das amostras, sem treinamento de modelos ou cálculo de desempenho.

In [37]:
# Verificação do alinhamento da divisão externa
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
print('Demonstrações de reamostragem concluídas.')

Demonstrações de reamostragem concluídas.


## Encerramento

Foram demonstradas divisão treino/teste, K-fold, K-fold repetido e bootstrap na HEALTH INSURANCE. Não há seleção de modelos ou cálculo de desempenho nesta etapa.

Para modelagem, ajustar o pré-processamento dentro de cada amostra de treino e aplicar à validação correspondente. No bootstrap, ajustar nas linhas sorteadas com suas repetições. O teste externo permanece reservado.

Os últimos conjuntos X_boot e X_oob são apenas a última repetição, não um conjunto final escolhido.